# Lab 1 - Your first agent in the Console

**Estimated time:** 10 min  ·**Estimated cost:** ~ $0.01-$0.02

> **Lab 1 is done in the Console (no code required).** This notebook is your step-by-step companion: follow the Console steps below on screen, then use the optional cells at the end to verify the result with Python.

## Goal
Build a coding-assistant agent entirely in the Anthropic Console: give it a model and the built-in toolset, point it at a cloud environment, start a session, and ask it to write and run a small Python script. Watching the agent decide to **write** a file, **run** it with bash, and **read** the result back makes the agent loop concrete before you ever touch code.

## Prerequisites
- An **Anthropic account** at `platform.claude.com`.
- A **workspace** selected, with an **API key** created (you won't paste it in the Console, but the workspace needs one).
- The **Console open** in your browser.

> Walk through the Chapter 2 setup lesson first (account, spend cap, key). The low spend cap makes labs like this stress-free.

## Console steps

Every step is a click or a typed message in the browser.

### Step 1 - Create the agent
1. In the left nav, open **Agents -> Create agent**.
2. **Name:** `My First Agent`.
3. **Model:** pick `claude-haiku-4-5-20251001` (the course default) from the dropdown.
4. **System prompt:**
   ```
   You are a helpful coding assistant. Write clean, well-documented code.
   When asked to create a file, write it under /workspace/, run it if
   appropriate, and verify the output.
   ```
5. **Tools:** leave the full **built-in agent toolset** enabled (this is the default `agent_toolset_20260401`; it includes `write`, `bash`, and `read`).
6. **MCP servers / Skills:** leave empty for now.
7. Save the agent.

> Tip: after saving, click **Edit agent**, then click the **code icon** in the upper right to peek at the code the Console generates. That request is exactly what Lab 2 builds by hand.

### Step 2 - Create or select a cloud environment
1. When prompted for an environment, choose **Create environment** (or pick an existing one).
2. **Type:** cloud.
3. **Networking:** `unrestricted` is fine for this lab.
4. No package presets needed; Python is already in the default cloud image.
5. Name it `quickstart-env` and save.

### Step 3 - Start a session
1. Click **Start session**.
2. Title the session `Fibonacci`.
3. You land in the session viewer with an inline chat box.

### Step 4 - Send the first message
In the chat box, send exactly this:

> Write fib.py printing the first 20 Fibonacci numbers, then run it

### Step 5 - Watch the tool calls
In the event stream / session viewer you should see, roughly in order:
- A short message where Claude explains what it's about to do.
- A **`write`** tool call creating `/workspace/fib.py`.
- A **`bash`** tool call running the script (e.g. `python /workspace/fib.py`).
- A **`read`** (or tool result) showing the script's output.
- A final message where Claude reports the numbers.

Watch the **session status** indicator move from `running` while the agent works to `idle` when the turn completes.

## Cost estimate
This is the one Console-only lab, so there is no local `session.id` for the notebook to meter. Use the Console's session details and billing view for the authoritative cost of the run.


## Expected output
- `/workspace/fib.py` exists in the session's environment.
- The agent's output shows the **first 20 Fibonacci numbers**: `0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89, 144, 233, 377, 610, 987, 1597, 2584, 4181`.
- The event stream shows `write`, `bash`, and `read` tool calls.
- The **session ends `idle`** (no errors, no stuck `running` state).

## Optional: verify it from Python

**This section is optional.** The lab itself is no-code: everything above happens in the Console. The cells below are a convenience so you can confirm from Python that your Console agent really exists. They are read-only listing calls (they do not create or modify anything). Skip them entirely if you just want the Console experience.

In [ ]:
# Verify that this notebook is using the uv-managed lab kernel.
import sys
from pathlib import Path

EXPECTED_KERNEL = "Managed Agents Labs (.venv)"
python_exe = Path(sys.executable)
python_prefix = Path(sys.prefix)

if ".venv" not in python_exe.parts and ".venv" not in python_prefix.parts:
    raise RuntimeError(
        f"Select the Jupyter kernel {EXPECTED_KERNEL!r} before running this notebook. "
        "From the repo root, run ./setup_uv.sh once to create and register it."
    )

print("Using uv environment:", python_prefix)

In [ ]:
import getpass
import os

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Enter your Anthropic API key: ")

print("ANTHROPIC_API_KEY configured for this notebook session.")

In [ ]:
# Optional verification: list your agents and confirm the Console agent is there.
# Read-only; creates nothing. Defensive on purpose so it never blocks the lab.
import os
from anthropic import Anthropic

BETAS = ["managed-agents-2026-04-01"]

if not os.environ.get("ANTHROPIC_API_KEY"):
    print("ANTHROPIC_API_KEY is not set, so this optional check is skipped.")
    print("That's fine - Lab 1 is a Console lab and needs no key here.")
else:
    try:
        client = Anthropic()
        agents = client.beta.agents.list(betas=BETAS)
        items = list(getattr(agents, "data", agents))
        if not items:
            print("No agents found yet. Make sure you saved the agent in Step 1,")
            print("and that this key belongs to the same workspace as the Console.")
        else:
            print(f"Found {len(items)} agent(s) in this workspace:\n")
            for a in items[:10]:
                name = getattr(a, "name", "(unnamed)")
                aid = getattr(a, "id", "(no id)")
                model = getattr(a, "model", "")
                print(f"  - {name}  [{aid}]  {model}")
            print("\nLook for 'My First Agent' in the list above to confirm your Console agent exists.")
    except Exception as e:
        print("Optional check could not complete (this does not affect the lab):")
        print(f"  {type(e).__name__}: {e}")
        print("Common causes: key from a different workspace, or no network in this kernel.")

## Bonus (optional): drive it with Claude Code

**Not required** - the Console steps above are the whole lab. If you want a taste of agentic engineering, open this repo in Claude Code and paste these plain-English prompts one at a time. Claude Code calls the Managed Agents API for you (the beta SDK adds the required beta header automatically).

1. **Create the agent and environment:**
   > Using the Anthropic Managed Agents beta SDK (`betas=['managed-agents-2026-04-01']`, all calls on `client.beta.*`), create a coding-assistant agent on `claude-haiku-4-5-20251001` with the built-in toolset `{'type': 'agent_toolset_20260401'}` enabled, plus a cloud environment with unrestricted networking. Print the agent id and environment id.

2. **Start a session and send the task:**
   > Start a session against that agent and environment, then send the message "Write fib.py printing the first 20 Fibonacci numbers, then run it" and stream the agent's reply to my terminal.

3. **Confirm the result:**
   > Tell me whether the session reached idle, which tools the agent used, and show me the Fibonacci numbers it printed.

## Stretch
- **Verify the artifact:** ask the agent, "Run /workspace/fib.py again and read the file back to confirm it has exactly 20 numbers." Watch it chain `bash` and `read`.
- **Add a docstring:** ask, "Add a module-level docstring and a docstring on the function explaining what it does, then re-run it." Notice the agent edits, re-writes, and re-runs without you touching anything.

## What you've learned
- How to create an agent, environment, and session entirely from the Console.
- How to read the event stream and follow `write` -> `bash` -> `read` tool calls.
- That a managed agent can write, execute, and verify code on its own, with the status moving `running` -> `idle`.
- That the same flow is one API request away (the Bonus path and Lab 2 prove it).